<a href="https://colab.research.google.com/github/saratyulyuova/tenge-ruble-network/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tenge and Ruble in the Emerging-Market Currency Network, 2015–2026
**Author:** Sara Tyulyuova

**Question:** How did the network position of the Kazakhstani tenge relative to the Russian ruble change after 24 February 2022?

**H₀:** the correlation-based distance between KZT and RUB is the same before and after 24 Feb 2022. **H₁:** it changed (two-sided).

Data preparation (download, cleaning, verification against official NBK and Bank of Russia rates) is in `network_analysis.ipynb`.

## 1. Data

In [8]:
!pip install -q networkx
import numpy as np, pandas as pd, matplotlib.pyplot as plt, networkx as nx, os
import requests, time, io
import xml.etree.ElementTree as ET
from scipy.stats import norm

BASE = "https://raw.githubusercontent.com/saratyulyuova/tenge-ruble-network/refs/heads/main/data"
prices = pd.read_csv(f"{BASE}/raw_prices.csv", index_col=0, parse_dates=True)
fac = pd.read_csv(f"{BASE}/factors.csv", index_col=0, parse_dates=True)
FIG = "figures"; os.makedirs(FIG, exist_ok=True)
print("prices:", prices.shape, "| factors:", fac.shape)

prices: (2909, 25) | factors: (2909, 3)


In [9]:
dates = pd.date_range("2015-09-02", "2026-09-30", freq="W-WED")
rows, failed = [], []
for i, d in enumerate(dates):
    url = f"https://nationalbank.kz/rss/get_rates.cfm?fdate={d:%d.%m.%Y}"
    try:
        r = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        for item in ET.fromstring(r.content).iter("item"):
            if item.findtext("title") == "USD":
                rows.append((d, float(item.findtext("description")) / float(item.findtext("quant"))))
                break
    except Exception:
        failed.append(d.date())
    if i % 100 == 0:
        print(f"{i} of {len(dates)}")
    time.sleep(0.2)
kzt_off = pd.Series(dict(rows)).sort_index()
print("KZT official:", len(kzt_off), "Wednesdays | failed:", len(failed))

0 of 579
100 of 579
200 of 579
300 of 579
400 of 579
500 of 579
KZT official: 574 Wednesdays | failed: 1


In [10]:
url = ("https://www.cbr.ru/scripts/XML_dynamic.asp"
       "?date_req1=01/08/2015&date_req2=30/09/2026&VAL_NM_RQ=R01235")
resp = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
cbr = pd.read_xml(io.BytesIO(resp.content), xpath=".//Record", parser="etree")
rub_off = pd.Series(
    cbr["Value"].astype(str).str.replace(",", ".").astype(float).values / cbr["Nominal"].values,
    index=pd.to_datetime(cbr["Date"], format="%d.%m.%Y"),
).sort_index()

def shift_back(s):
    """Bank of Russia sets the rate for day X on the previous business day."""
    s2 = s.copy(); s2.index = s2.index - pd.offsets.BDay(1)
    return s2[~s2.index.duplicated(keep="first")].sort_index()

print("RUB official:", len(rub_off), "days")

RUB official: 2758 days


In [11]:
# replace Yahoo KZT and RUB with official rates (decision from data verification)
prices_final = prices.copy()
prices_final["KZT"] = kzt_off.reindex(prices_final.index, method="ffill")
prices_final["RUB"] = shift_back(rub_off).reindex(prices_final.index, method="ffill")

def to_weekly_returns(p):
    w = p.loc["2015-08-26":].resample("W-WED").last().drop(columns="GOLD")
    return (-np.log(w).diff()).dropna()          # "+" = currency strengthened vs USD

R = to_weekly_returns(prices_final)               # main version (official KZT, RUB)
R_yahoo = to_weekly_returns(prices)               # all-Yahoo version, for robustness
F = np.log(fac.loc["2015-08-26":].resample("W-WED").last()).diff().dropna()
R, F = R.align(F, join="inner", axis=0)
R_yahoo = R_yahoo.reindex(R.index)

BREAK = "2022-02-24"
pre, post = R.loc[:"2022-02-23"], R.loc[BREAK:]
F_pre, F_post = F.loc[pre.index], F.loc[post.index]
print("Weeks:", len(R), "| before:", len(pre), "| after:", len(post), "| currencies:", R.shape[1])

Weeks: 578 | before: 338 | after: 240 | currencies: 24
